# Tuning FLORIS using ModelFit

Demonstrate tuning of FLORIS to the SCADA data using the ModelFit object

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from optuna.visualization.matplotlib import (
    plot_contour,
    plot_slice,
)

from flasc.data_processing.dataframe_manipulations import (
    is_day_or_night,
)
from flasc.model_fitting.cost_library import CostFunctionBase, TurbinePowerMeanAbsoluteError
from flasc.model_fitting.model_fit import extract_trial_data, ModelFit
from flasc.model_fitting.opt_library import opt_optuna
from flasc.utilities.utilities_examples import load_floris_smarteole

## Parameters

In [ ]:
# Number of trials per optimization
n_trials = 20

## Read data and prepare data

Use previously definied 60s data and condition the data as before

In [ ]:
# Read in data
root_path = Path.cwd()
f = root_path / "postprocessed" / "df_scada_data_60s_filtered_and_northing_calibrated.pkl"
df_scada = pd.read_pickle(f)

In [ ]:
# Add day/night boolean to SCADA data
latitude = 49.8435
longitude = 2.801556

# Compute day/night in default settings and plot
df_scada = is_day_or_night(df_scada, latitude, longitude)

In [ ]:
# Limit SCADA data to region of wake steering

# Specify offsets
start_of_offset = 200  # deg
end_of_offset = 240  # deg

# Limit SCADA to this region
df_scada = df_scada[
    (df_scada.wd_smarteole > (start_of_offset - 20))
    & (df_scada.wd_smarteole < (end_of_offset + 20))
]

In [ ]:
# Assign wd, ws and pow ref and subset SCADA based on reference variables used
# in the SMARTEOLE wake steering experiment (TODO reference the experiment)
df_scada = df_scada.assign(
    wd=lambda df_: df_["wd_smarteole"],
    ws=lambda df_: df_["ws_smarteole"],
    pow_ref=lambda df_: df_["pow_ref_smarteole"],
).reset_index(drop=True)

## Load FLORIS Model

In [ ]:
fm, _ = load_floris_smarteole(wake_model="emgauss")
D = fm.core.farm.rotor_diameters[0]

## Split the data into sub groups

In [ ]:
# Split SCADA into baseline and wake steeering (controlled)
df_scada_baseline = df_scada[df_scada.control_mode == "baseline"].reset_index(drop=True).copy()
df_scada_controlled = df_scada[df_scada.control_mode == "controlled"].reset_index(drop=True).copy()

In [ ]:
# Further split SCADA into day and night
df_scada_baseline_day = df_scada_baseline[df_scada_baseline.is_day].reset_index(drop=True).copy()
df_scada_baseline_night = df_scada_baseline[~df_scada_baseline.is_day].reset_index(drop=True).copy()
df_scada_controlled_day = (
    df_scada_controlled[df_scada_controlled.is_day].reset_index(drop=True).copy()
)
df_scada_controlled_night = (
    df_scada_controlled[~df_scada_controlled.is_day].reset_index(drop=True).copy()
)

## Tuning recovery on baseline data

The first element of the wake expansion parameter array is named we_1 in this analysis.  It governs the wake expansion in the empirical gaussian model up to the first defined breakpoint (often 10D).  Given the close spacing it is only necessary to tune this parameter.

First define the parameter for model fit

In [ ]:
# Just tune the first wake expansion parameter
parameter_list = [
    (
        "wake",
        "wake_velocity_parameters",
        "empirical_gauss",
        "wake_expansion_rates",
    )
]

parameter_name_list = [
    "we_1",
]

parameter_range_list = [
    (0.0, 0.05),
]

parameter_index_list = [0]

## Custom cost function

In [ ]:
# ModelFit provides a number of pre-defined cost functions in the cost_library module
# However, there is also flexibility to define custom cost function. Custom cost functions 
# should inherit from CostFunctionBase and implement the cost() method.

# In this example, we define a custom cost function that compute the absolute error of the power
# for a single turbine (turbine 004) and use this for model tuning.

class Turbine004PowerErrorAbs(CostFunctionBase):
    """Custom cost function to compute the absolute error of turbine 004 power."""
    def cost(self, df_floris):
        """Compute the absolute error of turbine 004 power."""
        return (self.df_scada["pow_004"] - df_floris["pow_004"]).abs().mean()
    
# Note that the provided TurbinePowerMeanAbsoluteError cost function can also be used for this
# purpose
cost_function_demo = TurbinePowerMeanAbsoluteError(turbine_power_subset=[4])

# Baseline tuning

Tune we_1 to the best fit for all the baseline data

In [ ]:
# We can also optionally df_scada_baseline to the cost function instantiation. We don't need to
# though, since it will be passed within the instantiation of ModelFit.
t004_cost_function = Turbine004PowerErrorAbs()

# Model Fit object
mf = ModelFit(
    df_scada_baseline,
    fm,
    t004_cost_function,
    parameter_list=parameter_list,
    parameter_name_list=parameter_name_list,
    parameter_range_list=parameter_range_list,
    parameter_index_list=parameter_index_list,
)

In [ ]:
# Run the optimization for n_trials
opt_result, study = opt_optuna(mf, timeout=None, n_trials=n_trials)

In [ ]:
# Print the best result
we_1_baseline = opt_result["parameter_values"][0]

print(f"Best we_1 from baseline tuning: {we_1_baseline}")


In [ ]:
# Use the slice plot provided by optuna to visualize the values of the parameters
# considered within the optimization
plot_slice(study)

# Tune the recovery to baseline day and night data

Using the data split into day and night results, tune the recovery to each to see how the best fit changes under these different conditions.

#### Daytime result

In [ ]:
mf_day = ModelFit(
    df_scada_baseline_day,
    fm,
    t004_cost_function,
    parameter_list=parameter_list,
    parameter_name_list=parameter_name_list,
    parameter_range_list=parameter_range_list,
    parameter_index_list=parameter_index_list,
)

opt_result_day, study_day = opt_optuna(mf_day, timeout=None, n_trials=n_trials)

we_1_baseline_day = opt_result_day["parameter_values"][0]

print(f"Best we_1 for day: {we_1_baseline_day}")


#### Night results

In [ ]:
mf_night = ModelFit(
    df_scada_baseline_night,
    fm,
    t004_cost_function,
    parameter_list=parameter_list,
    parameter_name_list=parameter_name_list,
    parameter_range_list=parameter_range_list,
    parameter_index_list=parameter_index_list,
)

opt_result_night, study_night = opt_optuna(mf_night, timeout=None, n_trials=n_trials)

we_1_baseline_night = opt_result_night["parameter_values"][0]

print(f"Best we_1 for night: {we_1_baseline_night}")

### Comparing the results

Notice the tuning parameters indicate less wake recovery in the night (smaller we_1) and more wake recovery in the day (larger we_1).

In [ ]:
# Print a small table with the best parameters
print("Best parameters:")
print(f"All data: {we_1_baseline:.3f}")
print(f"Day data: {we_1_baseline_day:.3f}")
print(f"Night data: {we_1_baseline_night:.3f}")


### Plot comparison

In [ ]:
# Compare the results for we_1 in a single plot
fig, ax = plt.subplots(figsize=(8, 5))

# Get data for each study
all_data_params, all_data_costs, best_all_param, best_all_cost = extract_trial_data(study, "we_1")
day_params, day_costs, best_day_param, best_day_cost = extract_trial_data(study_day, "we_1")
night_params, night_costs, best_night_param, best_night_cost = extract_trial_data(
    study_night, "we_1"
)

# Create plots
ax.plot(all_data_params, all_data_costs, label="All data", marker="o", color="black")
ax.plot(day_params, day_costs, label="Day data", marker="o", color="orange")
ax.plot(night_params, night_costs, label="Night data", marker="o", color="violet")

ax.axvline(best_all_param, color="black", ls="--")
ax.axvline(best_day_param, color="orange", ls="--")
ax.axvline(best_night_param, color="violet", ls="--")

ax.set_title("we_1 Parameter Tuning Comparison")
ax.set_ylabel("Cost / min(Cost)")
ax.set_xlabel("we_1")
ax.legend()
ax.grid(True, alpha=0.3)

# Tune the deflection gain to the baseline data

The deflection gain in the emgauss model defines the gain in the deflection of the wake at the downstream turbine following a yaw angle misalignment.  A higher gain indicates more deflection of the wake.

## Update the FLORIS model with the best values of we_1 for each case from the baseline data

In [ ]:
fm_baseline = fm.copy()
fm_baseline_day = fm.copy()
fm_baseline_night = fm.copy()

fm_baseline.set_param(parameter_list[0], we_1_baseline, 0)
fm_baseline_day.set_param(parameter_list[0], we_1_baseline_day, 0)
fm_baseline_night.set_param(parameter_list[0], we_1_baseline_night, 0)

## Tune the deflection parameter

First define the deflection parameter

In [ ]:
# Just tune the first wake expansion parameter
parameter_list = [
    (
        "wake",
        "wake_deflection_parameters",
        "empirical_gauss",
        "horizontal_deflection_gain_D",
    )
]

parameter_name_list = [
    "deflection_gain",
]

parameter_range_list = [
    (0.0, 5.0),
]

parameter_index_list = [None]

## Also define the yaw angles to be resimulated in each case

In [ ]:
# Set the yaw angle matrix for when using all the controlled data
yaw_vec = df_scada_controlled.wind_vane_005
yaw_angles_controlled = np.zeros((yaw_vec.shape[0], 7))
yaw_angles_controlled[:, 5] = yaw_vec  # Turbine 005 is the turbine implementing wake steering

# Set the yaw angles for day and night data
yaw_vec_day = df_scada_controlled_day.wind_vane_005
yaw_angles_controlled_day = np.zeros((yaw_vec_day.shape[0], 7))
yaw_angles_controlled_day[:, 5] = yaw_vec_day

yaw_vec_night = df_scada_controlled_night.wind_vane_005
yaw_angles_controlled_night = np.zeros((yaw_vec_night.shape[0], 7))
yaw_angles_controlled_night[:, 5] = yaw_vec_night

## Tune the deflection parameter

In [ ]:
# Tune to all controlled data
mf_deflection = ModelFit(
    df_scada_controlled,
    fm_baseline,  # Use the model tuned to all baseline data
    t004_cost_function,
    parameter_list=parameter_list,
    parameter_name_list=parameter_name_list,
    parameter_range_list=parameter_range_list,
    parameter_index_list=parameter_index_list,
    yaw_angles=yaw_angles_controlled,
)

opt_result_deflection, study_deflection = opt_optuna(
    mf_deflection, timeout=None, n_trials=n_trials
)
def_gain = opt_result_deflection["parameter_values"][0]

In [ ]:
# Tune to the day data
mf_day = ModelFit(
    df_scada_controlled_day,
    fm_baseline_day,  # Use the model tuned to day-time baseline data
    t004_cost_function,
    parameter_list=parameter_list,
    parameter_name_list=parameter_name_list,
    parameter_range_list=parameter_range_list,
    parameter_index_list=parameter_index_list,
    yaw_angles=yaw_angles_controlled_day,
)

opt_result_deflection_day, study_deflection_day = opt_optuna(
    mf_day, timeout=None, n_trials=n_trials
)
def_gain_day = opt_result_deflection_day["parameter_values"][0]


In [ ]:
# Tune to the night data
mf_night = ModelFit(
    df_scada_controlled_night,
    fm_baseline_night,  # Use the model tuned to night-time baseline data
    t004_cost_function,
    parameter_list=parameter_list,
    parameter_name_list=parameter_name_list,
    parameter_range_list=parameter_range_list,
    parameter_index_list=parameter_index_list,
    yaw_angles=yaw_angles_controlled_night,
)
opt_result_deflection_night, study_deflection_night = opt_optuna(
    mf_night, timeout=None, n_trials=n_trials
)
def_gain_night = opt_result_deflection_night["parameter_values"][0]

# Compare the results for the deflection gain

Results indicate that the wake deflection gain is larger at night than at day.  

In [ ]:
# Print a table with the best parameters
print("Best parameters:")
print(f"All data: {def_gain:.3f}")
print(f"Day data: {def_gain_day:.3f}")
print(f"Night data: {def_gain_night:.3f}")

In [ ]:
# Make comparison plot of the results

# Compare the results for deflection gain in a single plot
fig, ax = plt.subplots(figsize=(8, 5))

# Get data for each study
all_data_params, all_data_costs, best_all_param, best_all_cost = extract_trial_data(
    study_deflection, "deflection_gain"
)
day_params, day_costs, best_day_param, best_day_cost = extract_trial_data(
    study_deflection_day, "deflection_gain"
)
night_params, night_costs, best_night_param, best_night_cost = extract_trial_data(
    study_deflection_night, "deflection_gain"
)

# Create plots
ax.plot(all_data_params, all_data_costs, label="All data", marker="o", color="black")
ax.plot(day_params, day_costs, label="Day data", marker="o", color="orange")
ax.plot(night_params, night_costs, label="Night data", marker="o", color="violet")

ax.axvline(best_all_param, color="black", ls="--")
ax.axvline(best_day_param, color="orange", ls="--")
ax.axvline(best_night_param, color="violet", ls="--")

ax.set_title("Deflection Gain Tuning Comparison")
ax.set_ylabel("Cost / min(Cost)")
ax.set_xlabel("Deflection Gain")
ax.legend()
ax.grid(True, alpha=0.3)

# Simultaneously tune the we_1 and deflection gain parameters

As a final analysis, see what results would have been for all data case if we_1 and deflection were tuned simulaneously to the data

Note results would not be expected to be exactly the same because the data used in training is now not the same (instead of tuning expansion to baseline data and deflection_gain to controlled data, we are tuning both to all data)

In [ ]:
# Redo parameter list to include both we_1 and deflection_gain
parameter_list = [
    (
        "wake",
        "wake_velocity_parameters",
        "empirical_gauss",
        "wake_expansion_rates",
    ),
    (
        "wake",
        "wake_deflection_parameters",
        "empirical_gauss",
        "horizontal_deflection_gain_D",
    ),
]

parameter_name_list = [
    "we_1",
    "deflection_gain",
]

parameter_range_list = [
    (0.0, 0.05),
    (0.0, 5.0),
]

parameter_index_list = [0, None]

In [ ]:
# Redo yaw angle matrix to all data
yaw_vec = df_scada.wind_vane_005
yaw_angles_all = np.zeros((yaw_vec.shape[0], 7))
yaw_angles_all[:, 5] = yaw_vec


In [ ]:
mf_simultaneous = ModelFit(
    df_scada,
    fm,
    t004_cost_function,
    parameter_list=parameter_list,
    parameter_name_list=parameter_name_list,
    parameter_range_list=parameter_range_list,
    parameter_index_list=parameter_index_list,
    yaw_angles=yaw_angles_all,
)

# Double the number of trials since tuning 2 parameters
opt_result_simultaneous, study_simultaneous = opt_optuna(
    mf_simultaneous, timeout=None, n_trials=n_trials * 2
)


In [ ]:
# Show the contour plot using the study object
plot_contour(study_simultaneous)

In [ ]:
# Get the best parameters
we_1_simultaneous = opt_result_simultaneous["parameter_values"][0]
def_gain_simultaneous = opt_result_simultaneous["parameter_values"][1]


# Compare to the values computed sequentially
print("Sequential tuning results:")
print(f"Sequential tuning: {we_1_baseline:.3f}, {def_gain:.3f}")
print(f"Simultaneous tuning: {we_1_simultaneous:.3f}, {def_gain_simultaneous:.3f}")